In [1]:
class TrieNode:
  def __init__(self):
    self.children = {}
    self.is_end_of_username = False
    self.user_id = None

In [5]:
class UserAutocompleteTrie:
  def __init__(self):
    self.root = TrieNode()

  def insert(self, username, user_id):
    node = self.root
    for char in username:
      if char not in node.children:
        node.children[char] = TrieNode()
      node = node.children[char]
    node.is_end_of_username = True
    node.user_id = user_id

  def search(self, username):
    node = self.root
    for char in username:
      if char not in node.children:
        return None
      node = node.children[char]
    return node.user_id if node.is_end_of_username else None

  def starts_with(self, prefix):
    node = self.root
    for char in prefix:
      if char not in node.children:
        return False
      node = node.children[char]
    return True

  def autocomplete(self, prefix,  max_results=10):
    node = self.root
    for char in prefix:
      if char not in node.children:
        return []
      node = node.children[char]
    results = []
    def find_all(curr_node, curr_name):
      if len(results) >= max_results:
        return
      if curr_node.is_end_of_username:
        results.append((curr_name, curr_node.user_id))
      for char, child_node in sorted(curr_node.children.items()):
        find_all(child_node, curr_name + char)
    find_all(node, prefix)
    return results

  def count_words(self, node):
    count = 1 if node.is_end_of_username else 0
    for child in node.children.values():
      count += self.count_words(child)
    return count

  def get_height(self, node):
    if not node.children:
      return 0
    max_h = 0
    for child in node.children.values():
      h = self.get_height(child)
      if h > max_h:
        max_h = h
    return 1 + max_h

  def get_total_nodes(self, node):
    total = 1
    for child in node.children.values():
      total += self.get_total_nodes(child)
    return total

  def delete(self, username):
    curr = self.root
    for char in username:
      if char not in curr.children:
        return
      curr = curr.children[char]
    if curr.is_end_of_username:
      curr.is_end_of_username = False
      curr.user_id = None

In [3]:
class SegmentNode:
  def __init__(self):
    self.sum_value = 0
    self.max_value = 0
    self.min_value = 0
    self.left_child = None
    self.right_child = None

In [6]:
class ActivitySegmentTree:
  def __init__(self, activity_array):
    self.n = len(activity_array)
    self.root = self.build(activity_array, 0, self.n - 1)

  def build(self, activity_array, start, end):
    if start == end:
      node = SegmentNode()
      node.sum_value = activity_array[start]
      node.max_value = activity_array[start]
      node.min_value = activity_array[start]
      return node
    mid = (start + end) // 2
    node = SegmentNode()
    node.left_child = self.build(activity_array, start, mid)
    node.right_child = self.build(activity_array, mid + 1, end)
    node.sum_value = node.left_child.sum_value + node.right_child.sum_value
    node.max_value = max(node.left_child.max_value, node.right_child.max_value)
    node.min_value = min(node.left_child.min_value, node.right_child.min_value)
    return node

  def query(self, node, start, end, l, r):
    if r < start or end < l:
      return 0
    if l <= start and end <= r:
      return node.sum_value
    mid = (start + end) // 2
    return self.query(node.left_child, start, mid, l, r) + self.query(node.right_child, mid + 1, end, l, r)

  def get_range_max(self, node, start, end, l, r):
    if r < start or end < l:
      return float('-inf')
    if l <= start and end <= r:
      return node.max_value
    mid = (start + end) // 2
    return max(self.get_range_max(node.left_child, start, mid, l, r), self.get_range_max(node.right_child, mid + 1, end, l, r))

  def get_range_min(self, node, start, end, l, r):
    if r < start or end < l:
      return float('inf')
    if l <= start and end <= r:
      return node.min_value
    mid = (start + end) // 2
    return min(self.get_range_min(node.left_child, start, mid, l, r), self.get_range_min(node.right_child, mid + 1, end, l, r))

  def get_tree_size(self):
    return 2 * self.n - 1

  def get_height(self, node):
    if node is None or (node.left_child is None and node.right_child is None):
      return 0
    return 1 + max(self.get_height(node.left_child), self.get_height(node.right_child))

  def get_total_nodes(self):
    return 2* self.n - 1

  def get_leaf_values(self):
    results = []
    def traverse_leaves(node):
      if node is None:
        return
      if node.left_child is None and node.right_child is None:
        results.append(node.sum_value)
        return
      traverse_leaves(node.left_child)
      traverse_leaves(node.right_child)
    traverse_leaves(self.root)
    return results